# SaludAI - Consultas con el Agente

Este notebook demuestra como usar el agente de SaludAI para responder preguntas clinicas en lenguaje natural.

El agente:
1. Interpreta la pregunta del usuario
2. Resuelve terminologia clinica (SNOMED CT, CIE-10, LOINC)
3. Construye y ejecuta queries FHIR
4. Analiza los resultados con codigo Python (sandboxed)
5. Genera una respuesta narrativa con fuentes

## Prerequisitos

```bash
docker compose up -d          # HAPI FHIR con datos argentinos
cp .env.example .env          # Configurar API keys (Anthropic/OpenAI)
```

## 1. Configurar el agente

El agente necesita un LLM (Anthropic, OpenAI, u Ollama), un cliente FHIR, y opcionalmente Langfuse para tracing.

In [ ]:
import sys

sys.stdout.reconfigure(encoding="utf-8")

from dotenv import load_dotenv

load_dotenv()  # Cargar variables de .env

from saludai_agent.config import AgentConfig
from saludai_agent.llm import create_llm_client
from saludai_agent.loop import AgentLoop
from saludai_agent.tracing import create_tracer
from saludai_core.config import FHIRConfig
from saludai_core.fhir_client import FHIRClient
from saludai_core.locales import load_locale_pack
from saludai_core.terminology import TerminologyResolver

# Configuracion
fhir_config = FHIRConfig(fhir_server_url="http://localhost:8080/fhir")
agent_config = AgentConfig()  # Lee de variables de entorno (SALUDAI_*)

print(f"LLM provider: {agent_config.llm_provider}")
print(f"LLM model: {agent_config.llm_model}")
print(f"Max iterations: {agent_config.agent_max_iterations}")
print(f"Langfuse enabled: {agent_config.langfuse_enabled}")

## 2. Helper para ejecutar consultas

Creamos una funcion auxiliar que inicializa el agente y ejecuta una consulta, mostrando los resultados de forma legible.

In [ ]:
async def ask(query: str) -> None:
    """Ejecutar una consulta al agente y mostrar los resultados."""
    llm = create_llm_client(agent_config)
    resolver = TerminologyResolver()
    locale = load_locale_pack("ar")
    tracer = create_tracer(agent_config)

    async with FHIRClient(config=fhir_config) as client:
        loop = AgentLoop(
            llm=llm,
            fhir_client=client,
            terminology_resolver=resolver,
            config=agent_config,
            tracer=tracer,
            locale_pack=locale,
        )
        result = await loop.run(query)

    print(f"Pregunta: {result.query}")
    print(f"{'=' * 60}")
    print(f"\nRespuesta:\n{result.answer}")
    print(f"\n{'=' * 60}")
    print(f"Iteraciones: {result.iterations}")
    print(f"Tools usados: {len(result.tool_calls_made)}")
    for tc in result.tool_calls_made:
        print(f"  - {tc.name}({tc.arguments})")
    if result.trace_url:
        print(f"\nTrace en Langfuse: {result.trace_url}")

## 3. Consultas de ejemplo

### Consulta simple: Conteo de pacientes

In [ ]:
await ask("¿Cuantos pacientes hay registrados en el sistema?")

### Consulta media: Filtrando por condicion clinica y demografia

In [ ]:
await ask("¿Cuantos pacientes con diabetes tipo 2 son mayores de 60 anos?")

### Consulta compleja: Analisis cross-resource

Estas consultas requieren navegar multiples recursos FHIR y procesar datos con codigo.

In [ ]:
await ask("¿Cual es la condicion clinica mas frecuente entre los pacientes del sistema?")

In [ ]:
await ask("¿Que medicaciones toman los pacientes con hipertension?")

## 4. Consulta personalizada

Modifica la pregunta a continuacion para probar tu propia consulta:

In [ ]:
await ask("¿Cuantas mujeres mayores de 40 tienen asma?")

---

**Siguiente:** En el [notebook 03](03-benchmark-eval.ipynb) evaluamos la precision del agente con el benchmark FHIR-AgentBench.